# UFC Fight Prediction Modeling

This notebook implements a comprehensive machine learning pipeline for predicting UFC fight outcomes. It includes:

1.  **Standard ML Models:** XGBoost, Logistic Regression, Random Forest.
2.  **Simple Neural Network:** A standard feed-forward MLP using One-Hot Encoding.
3.  **ResNet with Embeddings:** A custom Residual Network that handles categorical variables via learned embeddings and numeric variables via a residual stream.
4.  **Visualization:** Architecture diagrams and performance evaluation.

## 1. Imports and Configuration

In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import copy
import io
import os

# Ensure plots display inline in Jupyter
%matplotlib inline

# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


## 2. Model Architectures
Here we define the PyTorch neural network architectures.

In [13]:
# ==========================================
# 1. MODEL ARCHITECTURES
# ==========================================

# --- A. Simple Feed-Forward Network ---
class SimpleUFCNet(nn.Module):
    def __init__(self, input_dim):
        super(SimpleUFCNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# --- B. ResNet with Embeddings ---
class ResidualBlock(nn.Module):
    def __init__(self, features, dropout_rate=0.3):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(features, features),
            nn.BatchNorm1d(features),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(features, features),
            nn.BatchNorm1d(features)
        )
        self.relu = nn.ReLU()
        
    def forward(self, x):
        residual = x
        out = self.block(x)
        out += residual # Skip Connection
        return self.relu(out)

class ResNetUFC(nn.Module):
    def __init__(self, num_numeric, cat_dims, embedding_dims, hidden_units=[256, 128]):
        super(ResNetUFC, self).__init__()
        
        # 1. Embedding Layers
        self.embeddings = nn.ModuleList([
            nn.Embedding(num, dim) for num, dim in zip(cat_dims, embedding_dims)
        ])
        total_emb_dim = sum(embedding_dims)
        
        # 2. Input Processing
        input_dim = num_numeric + total_emb_dim
        self.input_bn = nn.BatchNorm1d(num_numeric) 
        
        # 3. Initial Projection
        self.initial_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_units[0]),
            nn.BatchNorm1d(hidden_units[0]),
            nn.ReLU()
        )
        
        # 4. Residual Blocks
        self.res_blocks = nn.ModuleList()
        for i in range(len(hidden_units)-1):
            self.res_blocks.append(
                nn.Sequential(
                    ResidualBlock(hidden_units[i]),
                    nn.Linear(hidden_units[i], hidden_units[i+1]),
                    nn.BatchNorm1d(hidden_units[i+1]),
                    nn.ReLU()
                )
            )
            
        # 5. Output Head
        self.output = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(hidden_units[-1], 1),
            nn.Sigmoid()
        )
        
    def forward(self, x_num, x_cat):
        # Embeddings
        x_cat_list = []
        for i, emb in enumerate(self.embeddings):
            # Clamp indices to avoid out-of-range errors if new categories appear
            x_cat_safe = torch.clamp(x_cat[:, i], 0, emb.num_embeddings - 1)
            x_cat_list.append(emb(x_cat_safe))
        
        x_cat_combined = torch.cat(x_cat_list, 1)
        
        # Numeric
        x_num = self.input_bn(x_num)
        
        # Combine
        x = torch.cat([x_num, x_cat_combined], 1)
        
        # Forward Pass
        x = self.initial_layer(x)
        for block in self.res_blocks:
            x = block(x)
            
        return self.output(x)

## 3. Dataset Classes
Custom PyTorch Datasets for handling the data loading.

In [14]:
# ==========================================
# 2. DATASETS
# ==========================================

class SimpleDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1).to(device) if y is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return (self.X[idx], self.y[idx]) if self.y is not None else self.X[idx]

class ResNetDataset(Dataset):
    def __init__(self, X_num, X_cat, y=None):
        self.X_num = torch.tensor(X_num, dtype=torch.float32).to(device)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long).to(device)
        self.y = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1).to(device) if y is not None else None
    def __len__(self): return len(self.X_num)
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X_num[idx], self.X_cat[idx], self.y[idx]
        return self.X_num[idx], self.X_cat[idx]

## 4. The Predictor Engine
This class handles data preprocessing, training loops, evaluation, and prediction.

In [15]:
# ==========================================
# 3. PREDICTOR CLASS
# ==========================================

class UFCFightPredictor:
    def __init__(self):
        self.models = {}
        self.results = []
        
        # Preprocessors
        self.scaler_simple = StandardScaler()
        self.scaler_resnet = StandardScaler()
        self.label_encoders = {}
        
        # Metadata
        self.cat_cols = []
        self.num_cols = []
        self.training_cols_simple = [] # For OHE alignment
        self.cat_dims = []
        self.emb_dims = []
        
        # ML Config
        self.ml_models_config = {
            'XGBoost': XGBClassifier(n_estimators=500, learning_rate=0.01, max_depth=4, n_jobs=-1, random_state=42),
            'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
            'RandomForest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
            'MLP': MLPClassifier(hidden_layer_sizes=(128,64), max_iter=500, random_state=42),
            'SVC': SVC(class_weight='balanced', probability=True, random_state=42)
        }

    def _get_feature_lists(self, df):
        """Identify numeric and categorical columns dynamically."""
        drop_cols = ['Winner', 'BlueWin', 'RedFighter', 'BlueFighter', 'EmptyArena', 'Date', 'Location', 'Country', 
                     'Finish', 'FinishDetails', 'FinishRoundTime', 'TotalFightTime']
        
        exclude_cols = ['Winner', 'BlueWin', 'RedFighter', 'BlueFighter', 'Date', 'Location', 'Country', 'TitleBout', 'EmptyArena', 'WeightClass']
        leak_keywords = ['Finish', 'TotalFightTime', 'Referee', 'Round', 'weightRank', 'FPRank', 'WinsBy'] # stricter leakage filtering
        
        # candidates = [c for c in df.columns if c not in drop_cols]
        candidates = [c for c in df.columns if c not in exclude_cols and not any(kw in c for kw in leak_keywords)]
        cat_cols = [c for c in candidates if df[c].dtype == 'object']
        num_cols = [c for c in candidates if df[c].dtype != 'object']
        return num_cols, cat_cols

    def _preprocess_simple(self, df, fit=False):
        """Preprocessing for ML models and Simple NN (One-Hot Encoding)"""
        num_cols, cat_cols = self._get_feature_lists(df)
        
        # 1. Separate & Fill NaNs
        X_num = df[num_cols].fillna(0)
        X_cat = df[cat_cols].fillna('Unknown')
        
        # 2. One-Hot Encoding
        X_cat = pd.get_dummies(X_cat, drop_first=True)
        
        # 3. Combine
        X = pd.concat([X_num, X_cat], axis=1)
        
        # 4. Align Columns
        if fit:
            self.training_cols_simple = X.columns.tolist()
        else:
            X = X.reindex(columns=self.training_cols_simple, fill_value=0)
            
        # 5. Scale
        if fit:
            X = pd.DataFrame(self.scaler_simple.fit_transform(X), columns=X.columns)
        else:
            X = pd.DataFrame(self.scaler_simple.transform(X), columns=X.columns)
            
        return X

    def _preprocess_resnet(self, df, fit=False):
        """Preprocessing for ResNet (Label Encoding for Embeddings)"""
        num_cols, cat_cols = self._get_feature_lists(df)
        
        # 1. Numeric
        X_num = df[num_cols].fillna(0).values
        if fit:
            X_num = self.scaler_resnet.fit_transform(X_num)
        else:
            X_num = self.scaler_resnet.transform(X_num)
            
        # 2. Categorical (Label Encoding)
        X_cat = df[cat_cols].fillna('Unknown').copy()
        
        if fit:
            self.cat_dims = []
            self.emb_dims = []
            self.label_encoders = {}
            
            for col in cat_cols:
                le = LabelEncoder()
                X_cat[col] = le.fit_transform(X_cat[col].astype(str))
                self.label_encoders[col] = le
                
                # Embedding Sizing Rule
                n_classes = len(le.classes_)
                self.cat_dims.append(n_classes + 1) # +1 for unknown in future
                self.emb_dims.append(min(50, (n_classes + 1) // 2))
        else:
            for col in cat_cols:
                le = self.label_encoders.get(col)
                if le:
                    # Handle unseen labels by mapping them to 0 (or a new index if we allocated space)
                    # Here we map known labels to their index, unknowns to 0
                    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
                    X_cat[col] = X_cat[col].map(mapping).fillna(0).astype(int)
        
        return X_num, X_cat.values

    def train(self, csv_path):
        print(f"Loading data from {csv_path}...")
        df = pd.read_csv(csv_path)
        
        # Target
        y = df['Winner'].apply(lambda x: 1 if x == 'Blue' else 0)
        
        # --- A. Train ML & SimpleNN (OHE Data) ---
        print("\n=== Training ML & Simple NN (One-Hot Encoded Data) ===")
        X_simple = self._preprocess_simple(df, fit=True)
        X_train, X_test, y_train, y_test = train_test_split(X_simple, y, test_size=0.2, random_state=42, stratify=y)
        
        # 1. ML Models
        for name, model in self.ml_models_config.items():
            print(f"Training {name}...")
            model.fit(X_train, y_train)
            acc = accuracy_score(y_test, model.predict(X_test))
            self.models[name] = model
            self.results.append({'Model': name, 'Accuracy': acc})

        # 2. Simple NN
        print("Training Simple Neural Network...")
        train_ds = SimpleDataset(X_train.values, y_train)
        test_ds = SimpleDataset(X_test.values, y_test)
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
        
        simple_model = SimpleUFCNet(input_dim=X_train.shape[1]).to(device)
        opt = optim.Adam(simple_model.parameters(), lr=0.001)
        crit = nn.BCELoss()
        
        best_acc = 0
        for epoch in range(15):
            simple_model.train()
            for bx, by in train_loader:
                opt.zero_grad()
                loss = crit(simple_model(bx), by)
                loss.backward()
                opt.step()
            
            # Validation
            simple_model.eval()
            with torch.no_grad():
                preds = (simple_model(test_ds.X) > 0.5).float()
                acc = accuracy_score(test_ds.y.cpu(), preds.cpu())
                if acc > best_acc:
                    best_acc = acc
                    self.models['SimpleNN'] = copy.deepcopy(simple_model)
        
        self.results.append({'Model': 'SimpleNN', 'Accuracy': best_acc})
        
        # --- B. Train ResNet (Embedding Data) ---
        print("\n=== Training ResNet (Embedding + Numeric Data) ===")
        X_num, X_cat = self._preprocess_resnet(df, fit=True)
        
        # Split needs to match indices ideally, but random_split is easier here
        # We need to split X_num, X_cat, and y consistently
        indices = np.arange(len(y))
        train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)
        
        train_ds_res = ResNetDataset(X_num[train_idx], X_cat[train_idx], y.iloc[train_idx])
        test_ds_res = ResNetDataset(X_num[test_idx], X_cat[test_idx], y.iloc[test_idx])
        train_loader = DataLoader(train_ds_res, batch_size=32, shuffle=True)
        
        resnet_model_1 = ResNetUFC(num_numeric=X_num.shape[1], 
                                 cat_dims=self.cat_dims, 
                                 embedding_dims=self.emb_dims).to(device)
        opt_1 = optim.AdamW(resnet_model_1.parameters(), lr=0.001)

        resnet_model_2 = ResNetUFC(num_numeric=X_num.shape[1], 
                                 cat_dims=self.cat_dims, 
                                 embedding_dims=self.emb_dims,
                                 hidden_units=[128, 64]).to(device)
        opt_2 = optim.AdamW(resnet_model_2.parameters(), lr=0.001)

        resnet_model_3 = ResNetUFC(num_numeric=X_num.shape[1], 
                                 cat_dims=self.cat_dims, 
                                 embedding_dims=self.emb_dims,
                                 hidden_units=[64, 32]).to(device)
        opt_3 = optim.AdamW(resnet_model_3.parameters(), lr=0.001)
        resnet_models = {'ResNet_256_128': (resnet_model_1, opt_1),
                         'ResNet_128_64': (resnet_model_2, opt_2),
                         'ResNet_64_32': (resnet_model_3, opt_3)}

        
        best_acc = 0
        for name, (resnet_model, opt) in resnet_models.items():
            print(f"Training {name}...")
            for epoch in range(20):
                resnet_model.train()
                for bx_num, bx_cat, by in train_loader:
                    opt.zero_grad()
                    loss = crit(resnet_model(bx_num, bx_cat), by)
                    loss.backward()
                    opt.step()
                    
                # Validation
                resnet_model.eval()
                with torch.no_grad():
                    preds = (resnet_model(test_ds_res.X_num, test_ds_res.X_cat) > 0.5).float()
                    acc = accuracy_score(test_ds_res.y.cpu(), preds.cpu())
                    if acc > best_acc:
                        best_acc = acc
                        self.models[name] = copy.deepcopy(resnet_model)
                        
            self.results.append({'Model': name, 'Accuracy': best_acc})
        print("Training Complete.")

    def evaluate(self):
        return pd.DataFrame(self.results).sort_values(by='Accuracy', ascending=False)
    
    def predict_new(self, csv_path):
        """Predicts using the BEST performing model."""
        best_res = max(self.results, key=lambda x: x['Accuracy'])
        best_name = best_res['Model']
        model = self.models[best_name]
        print(f"Predicting with Best Model: {best_name} (Acc: {best_res['Accuracy']:.4f})")
        
        df = pd.read_csv(csv_path)
        
        if best_name.startswith('ResNet'):
            X_num, X_cat = self._preprocess_resnet(df, fit=False)
            model.eval()
            with torch.no_grad():
                t_num = torch.tensor(X_num, dtype=torch.float32).to(device)
                t_cat = torch.tensor(X_cat, dtype=torch.long).to(device)
                probs = model(t_num, t_cat).cpu().numpy().flatten()
        
        elif best_name == 'SimpleNN':
            X = self._preprocess_simple(df, fit=False)
            model.eval()
            with torch.no_grad():
                t_x = torch.tensor(X.values, dtype=torch.float32).to(device)
                probs = model(t_x).cpu().numpy().flatten()
                
        else: # ML Models
            X = self._preprocess_simple(df, fit=False)
            probs = model.predict_proba(X)[:, 1]
            
        res = df[['RedFighter', 'BlueFighter']].copy()
        res['PredictedWinner'] = ['Blue' if p > 0.5 else 'Red' for p in probs]
        res['BlueProb'] = probs
        return res

    def draw_architecture(self):
        """Draws a conceptual diagram of the Neural Networks."""
        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        
        # --- Draw Simple NN ---
        ax = axes[0]
        ax.set_title("Simple Neural Network Architecture", fontsize=15)
        ax.axis('off')
        
        layers = ["Input", "Linear(128)", "ReLU+Bn+Drop", "Linear(64)", "ReLU+Bn+Drop", "Output(1)"]
        y_pos = np.linspace(0.9, 0.1, len(layers))
        
        for i, (layer, y) in enumerate(zip(layers, y_pos)):
            rect = patches.FancyBboxPatch((0.3, y-0.04), 0.4, 0.08, boxstyle="round,pad=0.02", fc='skyblue', ec='black')
            ax.add_patch(rect)
            ax.text(0.5, y, layer, ha='center', va='center', fontsize=12, weight='bold')
            if i < len(layers)-1:
                ax.arrow(0.5, y-0.04, 0, -(y_pos[i]-y_pos[i+1]-0.08), head_width=0.02, fc='gray', ec='gray')

        # --- Draw ResNet ---
        ax = axes[1]
        ax.set_title("ResNet (UFC) Architecture", fontsize=15)
        ax.axis('off')
        
        blocks = ["Numeric Input", "Cat Inputs\n(Embeddings)", "Concat", "Linear Proj", 
                  "Residual Block 1\n(Skip Conn)", "Residual Block 2\n(Skip Conn)", "Output Head"]
        
        # Draw Numeric & Cat side by side
        ax.add_patch(patches.FancyBboxPatch((0.1, 0.85), 0.2, 0.1, boxstyle="round", fc='#ff9999', ec='black'))
        ax.text(0.2, 0.9, "Numeric", ha='center', va='center')
        
        ax.add_patch(patches.FancyBboxPatch((0.7, 0.85), 0.2, 0.1, boxstyle="round", fc='#99ff99', ec='black'))
        ax.text(0.8, 0.9, "Embeddings", ha='center', va='center')
        
        # Concat
        ax.add_patch(patches.FancyBboxPatch((0.3, 0.7), 0.4, 0.08, boxstyle="round", fc='lightgray', ec='black'))
        ax.text(0.5, 0.74, "Concatenate", ha='center', va='center')
        
        # Arrows to concat
        ax.arrow(0.2, 0.85, 0.2, -0.1, head_width=0.02, fc='black')
        ax.arrow(0.8, 0.85, -0.2, -0.1, head_width=0.02, fc='black')
        
        # ResBlocks
        y_curr = 0.6
        for block in ["Projection", "ResBlock 1", "ResBlock 2", "Output"]:
            ax.add_patch(patches.FancyBboxPatch((0.35, y_curr-0.05), 0.3, 0.1, boxstyle="round", fc='gold', ec='black'))
            ax.text(0.5, y_curr, block, ha='center', va='center', weight='bold')
            
            # Draw Skip Connection Curve for ResBlocks
            if "ResBlock" in block:
                style = "Simple,tail_width=0.5,head_width=4,head_length=8"
                kw = dict(arrowstyle=style, color="k")
                # Arc from top of box to bottom of box
                arc = patches.FancyArrowPatch((0.66, y_curr+0.02), (0.66, y_curr-0.02), 
                                              connectionstyle="arc3,rad=2.5", **kw)
                ax.add_patch(arc)
                ax.text(0.75, y_curr, "Skip +", fontsize=10, color='blue')
                
            ax.arrow(0.5, y_curr-0.05, 0, -0.05, head_width=0.02, fc='black')
            y_curr -= 0.15

        plt.show()

## 5. Execution
Run the training pipeline. 
**Note:** Ensure you have a CSV file named `ufc-master.csv` in your working directory.

In [16]:
# 1. Initialize
predictor = UFCFightPredictor()

# 2. Visualize Architectures
# predictor.draw_architecture()

# 3. Train All Models
# Make sure 'ufc-master.csv' is in your current directory or upload it if using Colab
if os.path.exists('ufc-master.csv'):
    predictor.train('ufc-master.csv')
    
    # 4. Compare Results
    print(predictor.evaluate())
else:
    print("Dataset 'ufc-master.csv' not found. Please upload the file.")

# 5. Predict New Data (Example usage)
if os.path.exists('upcoming.csv'):
    preds = predictor.predict_new('upcoming.csv')
    display(preds)

Loading data from ufc-master.csv...

=== Training ML & Simple NN (One-Hot Encoded Data) ===
Training XGBoost...
Training LogisticRegression...
Training RandomForest...
Training MLP...
Training SVC...
Training Simple Neural Network...

=== Training ResNet (Embedding + Numeric Data) ===
Training ResNet_256_128...
Training ResNet_128_64...
Training ResNet_64_32...
Training Complete.
                Model  Accuracy
8        ResNet_64_32  0.669977
0             XGBoost  0.666921
7       ResNet_128_64  0.663866
5            SimpleNN  0.662338
6      ResNet_256_128  0.662338
2        RandomForest  0.654698
1  LogisticRegression  0.651642
4                 SVC  0.644003
3                 MLP  0.572193
Predicting with Best Model: ResNet_64_32 (Acc: 0.6700)


,RedFighter,BlueFighter,PredictedWinner,BlueProb
0,Colby Covington,Joaquin Buckley,Blue,0.570441
1,Cub Swanson,Billy Quarantillo,Red,0.482523
2,Manel Kape,Bruno Silva,Red,0.160105
3,Vitor Petrino,Dustin Jacoby,Red,0.204298
4,Adrian Yanez,Daniel Marcos,Blue,0.671948
5,Navajo Stirling,Tuco Tokkos,Red,0.048589
6,Michael Johnson,Ottman Azaitar,Red,0.263093
7,Joel Alvarez,Drakkar Klose,Red,0.259043
8,Sean Woodson,Fernando Padilla,Red,0.186738
9,Miles Johns,Felipe Lima,Blue,0.675873
